# chesszero — étape 6e : self-play sur Kaggle

**Deux modes**, choisis dans la cellule 0 :
- `MODE = "bench"` (**par défaut**) : **essai de débit de 15 min**, rien n'est ajouté au tampon. Phase 1 (5 min, réglages
  réels : 3 processus × 64 parties) : simulations/s, GPU, CPU, taille des lots envoyés au GPU ; phase 2 (10 min,
  16 parties par processus, pour que des parties se terminent) : longueur des parties, fins, répétitions précoces.
  Le rapport finit par les parties par heure et **le nombre d'heures pour atteindre 250 000 positions d'entraînement**.
- `MODE = "full"` : **session complète** (`FULL_MINUTES`, 11 h 30 par défaut). Avec `SELFPLAY_ONLY = True` (défaut) :
  self-play seul, pour remplir le tampon. Avec `False` : boucle simple et provisoire (tranches de self-play, puis un tour
  d'entraînement dès que le tampon atteint 250 000 positions) ; l'arène et les ancres fixes arrivent à l'étape 7.

**Réglages de la session** (panneau de droite) :

| Réglage | Valeur |
|---|---|
| Accelerator | **GPU T4 x2** (un seul GPU est utilisé : le goulot est le CPU) ; à défaut GPU P100 |
| Internet | **On** (clone du dépôt, tables Syzygy) |
| Persistence | No persistence |

**Entrées** (*Add Input*) :
1. le Dataset du réseau de départ (`best.pt` de l'étape 4b, SHA-256 `c1f616c6…18b33` vérifié par la cellule 3) ;
2. le Dataset `lichess-24m` (seulement si `SELFPLAY_ONLY = False`) ;
3. **à partir de la 2e session** : la sortie de la session précédente de CE notebook (*Add Input → Your Work → ce notebook*) :
   la cellule 3 y trouve `selfplay/` (le tampon) et `sp_train/` (l'état d'entraînement) et les reprend.

**Sorties** (`/kaggle/working`, c'est ce qui survit à la session) : `selfplay/` (le tampon : les shards les plus récents
jusqu'à 500 000 positions + `sessions/*.json` = statistiques de chaque session), `sp_train/` (avec entraînement),
`bench_selfplay/` (essai de débit). Le dépôt et les tables sont dans `/tmp` : ils ne gonflent pas la sortie.

In [ ]:
# 0. Réglages ---------------------------------------------------------------------------------------------
MODE = "bench"            # "bench" = essai de débit (15 min, défaut) ; "full" = session complète
BENCH_MINUTES = 15        # bench : durée totale (5 min de débit + 10 min de parties)
FULL_MINUTES = 690        # full : budget de la session (11 h 30 ; Kaggle coupe à 12 h)
SELFPLAY_ONLY = True      # full : True = self-play seul (remplir le tampon) ; False = + tours d'entraînement (provisoire)
WORKERS = None            # processus de self-play (None = cœurs CPU - 1, au plus 3)
GAMES_PER_WORKER = None   # parties menées ensemble par processus (None = [selfplay] de configs/kaggle.toml : 64)
EXPECTED_SHA256 = "c1f616c6eb908a89645b611e61229828a900c09d40c3477f8ba0284b22418b33"  # best.pt de kaggle_sup (pas 92 000)

REPO_URL = "https://github.com/Enz091/Project_chess.git"
BRANCH = "main"

import os
import shutil
import subprocess
import sys
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
REPO_DIR = Path("/tmp/Project_chess")   # hors de /kaggle/working : pas dans la sortie
SYZYGY = Path("/tmp/syzygy")            # tables WDL (396 Mo), retéléchargées à chaque session
assert MODE in ("bench", "full"), MODE


def run(cmd, cwd=None):
    # Lance une commande et affiche sa sortie au fil de l'eau ; lève une erreur si elle échoue.
    proc = subprocess.Popen([str(c) for c in cmd], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"échec ({proc.returncode}) : {' '.join(str(c) for c in cmd)}")

In [ ]:
# 1. Dépôt et installation (sans réinstaller torch) -----------------------------------------------------------
import importlib
import importlib.metadata as md

torch_before = md.version("torch")
if (REPO_DIR / ".git").exists():
    run(["git", "pull", "--ff-only"], cwd=REPO_DIR)
else:
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])
run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR])  # pas de torch dans les dépendances
sys.path.insert(0, str(REPO_DIR / "src"))  # le noyau ne voit pas un paquet installé après son démarrage
importlib.invalidate_caches()
run(["git", "log", "-1", "--format=commit %h  %s"], cwd=REPO_DIR)
assert md.version("torch") == torch_before, "pip a modifié torch : à ne pas faire sur Kaggle"
print("torch inchangé :", torch_before)

In [ ]:
# 2. Environnement ------------------------------------------------------------------------------------------------
import platform

import torch

print("Python :", platform.python_version(), "| torch :", torch.__version__, "| CUDA :", torch.version.cuda)
cores = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
WORKERS = WORKERS or max(1, min(3, cores - 1))   # un cœur reste libre (processus principal, mesures)
print(f"Cœurs CPU : {cores} -> {WORKERS} processus de self-play")
if Path("/proc/meminfo").exists():
    mem = {l.split(":")[0]: int(l.split()[1]) for l in Path("/proc/meminfo").read_text().splitlines()[:3]}
    print(f"RAM : {mem['MemTotal'] / 1e6:.0f} Go (libre {mem['MemAvailable'] / 1e6:.0f} Go) ; "
          f"disque /kaggle/working : {shutil.disk_usage(WORKING).free / 1e9:.0f} Go libres")
if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU : Session options -> Accelerator -> GPU T4 x2")
p = torch.cuda.get_device_properties(0)
print(f"GPU : {p.name}, {p.total_memory / 1e9:.0f} Go, {torch.cuda.device_count()} visible(s) (le premier est utilisé)")

In [ ]:
# 3. Entrées : réseau de départ (SHA-256 vérifié), données Lichess, sorties des sessions précédentes ----------------
import hashlib


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


candidates = [p for p in INPUT_ROOT.glob("**/*.pt") if p.stat().st_size > 10_000_000 and "sp_train" not in p.parts]
INIT = next((p for p in candidates if sha256(p) == EXPECTED_SHA256), None)
if INIT is None:
    raise FileNotFoundError("Réseau de départ introuvable : ajouter le Dataset contenant best.pt (SHA-256 "
                            f"{EXPECTED_SHA256[:8]}…). Fichiers .pt vus : {[str(p) for p in candidates]}")
print("Réseau de départ :", INIT, "(SHA-256 conforme)")

HUMAN = next((p.parent for p in sorted(INPUT_ROOT.glob("**/train_00000.npz"))), None)
print("Données Lichess :", HUMAN or "absentes")
if MODE == "full" and not SELFPLAY_ONLY and HUMAN is None:
    raise FileNotFoundError("SELFPLAY_ONLY = False demande le Dataset lichess-24m (Add Input)")

# Tampon des sessions précédentes : dossiers « selfplay » qui contiennent des shards (.npz + .json)
PREVIOUS = sorted({p.parent for p in INPUT_ROOT.glob("**/selfplay/*.npz") if p.with_suffix(".json").exists()})
TRAIN_STATE = next((p.parent for p in sorted(INPUT_ROOT.glob("**/sp_train/train_state.pt"))), None)
print("Tampon des sessions précédentes :", [str(p) for p in PREVIOUS] or "aucun (premier remplissage)")
print("État d'entraînement précédent :", TRAIN_STATE or "aucun")

In [ ]:
# 4. Tables Syzygy WDL (3-4-5 pièces, miroir Lichess, chaque fichier vérifié par SHA-256) -----------------------------
run([sys.executable, REPO_DIR / "scripts" / "download_syzygy.py", "--out", SYZYGY, "--wdl-only"])

In [ ]:
# 5. Self-play : essai de débit ou session complète ----------------------------------------------------------------
cmd = [sys.executable, REPO_DIR / "scripts" / "selfplay_session.py", MODE, "--work", WORKING, "--init", INIT,
       "--syzygy", SYZYGY, "--workers", WORKERS]
if GAMES_PER_WORKER:
    cmd += ["--games-per-worker", GAMES_PER_WORKER]
if MODE == "bench":
    cmd += ["--minutes", BENCH_MINUTES]
else:
    cmd += ["--minutes", FULL_MINUTES]
    if PREVIOUS:
        cmd += ["--inputs", *PREVIOUS]
    if not SELFPLAY_ONLY:
        cmd += ["--with-training", "--human", HUMAN]
        if TRAIN_STATE:
            cmd += ["--train-state", TRAIN_STATE]
run(cmd)

In [ ]:
# 6. Contenu de la sortie ------------------------------------------------------------------------------------------
for d in sorted(p for p in WORKING.iterdir() if p.is_dir()):
    size = sum(f.stat().st_size for f in d.rglob("*") if f.is_file())
    print(f"{d.name:<16} {size / 1e6:8.1f} Mo  {sum(1 for _ in d.rglob('*.npz'))} shards")
print("\nÀ la session suivante : Add Input -> Your Work -> ce notebook (cette version), puis relancer.")